In [1]:
!pip install albumentations opencv-python lxml scikit-learn  -q

# Imports

In [2]:
# =========================
# Standard Libraries
# =========================
import os
import sys
import csv
import copy
import shutil
import random
import logging
from datetime import datetime
from pathlib import Path
from collections import defaultdict, Counter
from typing import List, Tuple

# =========================
# XML Processing
# =========================
import xml.etree.ElementTree as ET
from lxml import etree

# =========================
# Data & Numerical
# =========================
import numpy as np
import pandas as pd

# =========================
# Computer Vision
# =========================
import cv2
import albumentations as A

# =========================
# Deep Learning
# =========================
import torch
from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# =========================
# ML Utilities
# =========================
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from tabulate import tabulate

# augmenter_fixed_complete.py
import os
import cv2
import copy
import numpy as np
import itertools
import shutil
from datetime import datetime
import xml.etree.ElementTree as ET
from collections import defaultdict
import random
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# time
import time
MAX_TRAINING_TIME = int((9) * 60 * 60)  # 9 hours in seconds
print("="*10)
print(MAX_TRAINING_TIME)
print("="*10)

32400


# Logging

In [3]:

def setup_logger(log_dir="logs", log_name="train"):
    os.makedirs(log_dir, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"{log_name}_{timestamp}.log")

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler(sys.stdout)
        ]
    )

    return log_file

log_file = setup_logger()

import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Augmentation

In [4]:
class FixedAugmenter:
    """Complete fixed-augmentation class with high-diversity / low-cost transforms."""

    def __init__(self):
        self.transform_history = []

    # ---------- geometry helpers ----------
    @staticmethod
    def _clip_boxes(bboxes, w, h):
        out = []
        for bbox in bboxes:
            x1, y1, x2, y2 = bbox
            x1 = max(0, min(x1, w - 1))
            y1 = max(0, min(y1, h - 1))
            x2 = max(x1 + 1, min(x2, w))
            y2 = max(y1 + 1, min(y2, h))
            if x2 - x1 > 5 and y2 - y1 > 5:
                out.append([x1, y1, x2, y2])
        return out

    # ---------- individual transforms ----------
    def apply_horizontal_flip(self, image, bboxes):
        if image is None or len(image.shape) != 3:
            return image, bboxes
        w = image.shape[1]
        img = cv2.flip(image, 1)
        out = []
        for bbox in bboxes:
            if len(bbox) != 4:
                continue
            x1, y1, x2, y2 = bbox
            out.append([w - x2, y1, w - x1, y2])
        return img, self._clip_boxes(out, w, image.shape[0])

    def apply_vertical_flip(self, image, bboxes):
        if image is None or len(image.shape) != 3:
            return image, bboxes
        h = image.shape[0]
        img = cv2.flip(image, 0)
        out = []
        for bbox in bboxes:
            if len(bbox) != 4:
                continue
            x1, y1, x2, y2 = bbox
            out.append([x1, h - y2, x2, h - y1])
        return img, self._clip_boxes(out, image.shape[1], h)

    def apply_brightness_contrast(self, image):
        if image is None:
            return image
        alpha = np.random.uniform(0.8, 1.2)
        beta = np.random.randint(-30, 30)
        return np.clip(cv2.convertScaleAbs(image, alpha=alpha, beta=beta), 0, 255)

    def apply_noise(self, image):
        if image is None:
            return image
        noise = np.random.normal(0, 5, image.shape).astype(np.int16)
        return np.clip(image.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    def apply_blur(self, image):
        if image is None:
            return image
        k = random.choice([3, 5])
        return cv2.GaussianBlur(image, (k, k), 0)

    def apply_hsv_shift(self, image):
        if image is None or len(image.shape) != 3:
            return image
        try:
            hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)
            hsv[:, :, 0] = (hsv[:, :, 0] + np.random.uniform(-10, 10)) % 180
            hsv[:, :, 1] = np.clip(hsv[:, :, 1] * np.random.uniform(0.8, 1.2), 0, 255)
            hsv[:, :, 2] = np.clip(hsv[:, :, 2] * np.random.uniform(0.8, 1.2), 0, 255)
            return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        except:
            return image

    def apply_rotation(self, image, bboxes):
        if image is None:
            return image, bboxes
        h, w = image.shape[:2]
        angle = np.random.uniform(-10, 10)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        out = []
        for bbox in bboxes:
            if len(bbox) != 4:
                continue
            pts = np.array([[bbox[0], bbox[1]], [bbox[2], bbox[1]],
                            [bbox[2], bbox[3]], [bbox[0], bbox[3]]], dtype=np.float32)
            pts = (M @ np.column_stack([pts, np.ones(4)]).T).T
            xs, ys = pts[:, 0], pts[:, 1]
            x1, y1, x2, y2 = np.clip(xs.min(), 0, w), np.clip(ys.min(), 0, h), \
                np.clip(xs.max(), 0, w), np.clip(ys.max(), 0, h)
            if x2 - x1 > 5 and y2 - y1 > 5:
                out.append([x1, y1, x2, y2])
            else:
                out.append(bbox)
        return rotated, self._clip_boxes(out, w, h)

    def apply_scale(self, image, bboxes):
        if image is None:
            return image, bboxes
        h, w = image.shape[:2]
        scale = np.random.uniform(0.9, 1.1)
        new_w, new_h = int(w * scale), int(h * scale)
        if scale < 1.0:
            scaled = cv2.resize(image, (new_w, new_h))
            pad_x, pad_y = (w - new_w) // 2, (h - new_h) // 2
            canvas = np.zeros((h, w, 3), dtype=np.uint8)
            canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = scaled
            out = [[x * scale + pad_x, y * scale + pad_y, x2 * scale + pad_x, y2 * scale + pad_y]
                   for x, y, x2, y2 in bboxes]
            return canvas, self._clip_boxes(out, w, h)
        else:
            scaled = cv2.resize(image, (new_w, new_h))
            crop_x, crop_y = (new_w - w) // 2, (new_h - h) // 2
            cropped = scaled[crop_y:crop_y + h, crop_x:crop_x + w]
            out = [[max(0, x * scale - crop_x), max(0, y * scale - crop_y),
                    min(w, x2 * scale - crop_x), min(h, y2 * scale - crop_y)]
                   for x, y, x2, y2 in bboxes]
            return cropped, self._clip_boxes(out, w, h)

    def apply_translate(self, image, bboxes):
        if image is None:
            return image, bboxes
        h, w = image.shape[:2]
        dx = int(np.round(np.random.uniform(-0.06, 0.06) * w))
        dy = int(np.round(np.random.uniform(-0.06, 0.06) * h))
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        shifted = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        out = [[x + dx, y + dy, x2 + dx, y2 + dy] for x, y, x2, y2 in bboxes]
        return shifted, self._clip_boxes(out, w, h)

    def apply_cutout(self, image):
        if image is None:
            return image
        h, w = image.shape[:2]
        side = int(0.06 * min(h, w))
        img = image.copy()
        for _ in range(3):
            x1 = np.random.randint(0, w - side)
            y1 = np.random.randint(0, h - side)
            img[y1:y1 + side, x1:x1 + side] = np.random.randint(0, 255, (side, side, 3), dtype=np.uint8)
        return img

    # ---------- safe orchestrator ----------
    def safe_augment(self, image, bboxes, labels):
        if image is None:
            return None, [], [], []
        aug_img = image.copy()
        aug_boxes = copy.deepcopy(bboxes)
        aug_labels = copy.deepcopy(labels)
        methods = []

        transforms = []
        if np.random.rand() > 0.5:
            transforms.append(('hflip', lambda img, boxes: self.apply_horizontal_flip(img, boxes)))
        if np.random.rand() > 0.7:
            transforms.append(('vflip', lambda img, boxes: self.apply_vertical_flip(img, boxes)))
        if np.random.rand() > 0.6:
            transforms.append(('brightness', lambda img, boxes: (self.apply_brightness_contrast(img), boxes)))
        if np.random.rand() > 0.7:
            transforms.append(('noise', lambda img, boxes: (self.apply_noise(img), boxes)))
        if np.random.rand() > 0.8:
            transforms.append(('blur', lambda img, boxes: (self.apply_blur(img), boxes)))
        if np.random.rand() > 0.6:
            transforms.append(('hsv', lambda img, boxes: (self.apply_hsv_shift(img), boxes)))
        random.shuffle(transforms)
        num_apply = min(len(transforms), random.randint(1, 4))
        for i in range(num_apply):
            name, func = transforms[i]
            try:
                res = func(aug_img, aug_boxes)
                if isinstance(res, tuple) and len(res) == 2:          # image + boxes
                    aug_img, aug_boxes = res
                else:                                                   # single image
                    aug_img = res
                methods.append(name)
            except Exception:
                continue

        # validate
        if not isinstance(aug_img, np.ndarray):
            return None, [], [], []
        h, w = aug_img.shape[:2]
        valid_boxes, valid_labels = [], []
        for box, lab in zip(aug_boxes, aug_labels):
            if len(box) != 4:
                continue
            x1, y1, x2, y2 = self._clip_boxes([box], w, h)[0]
            if x2 - x1 > 5 and y2 - y1 > 5:
                valid_boxes.append([x1, y1, x2, y2])
                valid_labels.append(lab)
        return aug_img, valid_boxes, valid_labels, methods


# ------------------------------------------------------------------
# ---------------  top-level runner  -------------------------------
# ------------------------------------------------------------------
def run_fixed_augmentation(train_img_dir, train_xml_dir,
                           target_per_class=9000,
                           output_dir=None,
                           report_file="augmentation_report_fixed.txt"):
    """High-diversity fixed augmentation – complete version."""
    if output_dir:
        os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "annotations"), exist_ok=True)
        img_out_dir = os.path.join(output_dir, "images")
        xml_out_dir = os.path.join(output_dir, "annotations")
    else:
        img_out_dir, xml_out_dir = train_img_dir, train_xml_dir

    log_f = open(report_file, "w", encoding="utf-8")

    def log(msg):
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ln = f"[{ts}] {msg}"
        print(ln)
        log_f.write(ln + "\n")
        log_f.flush()

    log("=" * 70)
    log("FIXED AUGMENTATION – COMPLETE DIVERSE PIPELINE")
    log("=" * 70)

    # 1. analyse -------------------------------------------------------------
    def analyse(xml_dir):
        cls_cnt = defaultdict(int)
        img_info = {}
        all_cls = set()
        for xml in [f for f in os.listdir(xml_dir) if f.endswith(".xml")]:
            try:
                root = ET.parse(os.path.join(xml_dir, xml)).getroot()
                boxes, labels, cls_in_img = [], [], set()
                for obj in root.findall("object"):
                    cls = obj.find("name").text
                    b = obj.find("bndbox")
                    box = [float(b.find(t).text) for t in ("xmin", "ymin", "xmax", "ymax")]
                    boxes.append(box)
                    labels.append(cls)
                    cls_in_img.add(cls)
                    cls_cnt[cls] += 1
                    all_cls.add(cls)
                img_info[xml] = {"boxes": boxes, "labels": labels, "classes": cls_in_img, "num": len(boxes)}
            except Exception as e:
                log(f"Error parsing {xml}: {e}")
        return dict(cls_cnt), img_info, sorted(all_cls)

    init_counts, img_info, all_classes = analyse(train_xml_dir)
    log("\nInitial counts:")
    for c in all_classes:
        log(f"  {c}: {init_counts[c]}")

    # 2. needs ---------------------------------------------------------------
    needs = {c: max(0, target_per_class - init_counts[c]) for c in all_classes}
    log("\nAugmentation needs:")
    for c, n in sorted(needs.items()):
        log(f"  {c}: {n}")

    # 3. schedule ------------------------------------------------------------
    log("\nCreating schedule...")
    cls2imgs = defaultdict(list)
    for xml, info in img_info.items():
        for c in info["classes"]:
            cls2imgs[c].append(xml)

    schedule = []
    for c, need in sorted(needs.items(), key=lambda x: x[1], reverse=True):
        donors = cls2imgs[c]
        if not donors:
            log(f"Warning: no donor for {c}")
            continue
        random.shuffle(donors)
        max_per_donor = min(40, max(5, int(np.ceil(need / len(donors)))))
        remaining = need
        for xml in itertools.cycle(donors):
            if remaining <= 0:
                break
            num = min(random.randint(max_per_donor - 2, max_per_donor + 2), remaining)
            schedule.append((xml, c, num))
            remaining -= num
    log(f"Planned augmentations: {len(schedule)}")

    # 4. augment -------------------------------------------------------------
    augmenter = FixedAugmenter()
    curr_counts = init_counts.copy()
    total_created = 0

    log("\nStarting augmentation...")
    for xml_file, tgt_cls, num_aug in tqdm(schedule, desc="Augmenting"):
        if all(curr_counts[c] >= target_per_class for c in all_classes):
            log("Global target reached – stopping early.")
            break
        if curr_counts[tgt_cls] >= target_per_class:
            continue

        base = os.path.splitext(xml_file)[0]
        # find image
        img_path = None
        ext = None
        for e in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
            p = os.path.join(train_img_dir, base + e)
            if os.path.exists(p):
                img_path, ext = p, e
                break
        if not img_path:
            log(f"Image not found: {base}")
            continue
        img = cv2.imread(img_path)
        if img is None:
            log(f"Failed to load {img_path}")
            continue
        info = img_info.get(xml_file)
        if not info:
            continue
        boxes, labels = info["boxes"], info["labels"]

        for aug_idx in range(num_aug):
            if curr_counts[tgt_cls] >= target_per_class:
                break
            aug_img, aug_boxes, aug_labels, methods = augmenter.safe_augment(img, boxes, labels)
            if aug_img is None or len(aug_boxes) == 0:
                continue
            ts = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            new_name = f"{base}_aug_{tgt_cls}_{ts}_{aug_idx}"
            img_out_path = os.path.join(img_out_dir, new_name + ext)
            try:
                if not cv2.imwrite(img_out_path, aug_img):
                    raise IOError("imwrite failed")
            except Exception as e:
                log(f"Save image error {new_name}: {e}")
                continue

            # XML
            xml_out_path = os.path.join(xml_out_dir, new_name + ".xml")
            try:
                root = ET.Element("annotation")
                for tag, text in (("folder", os.path.basename(img_out_dir)),
                                  ("filename", new_name + ext),
                                  ("path", img_out_path)):
                    ET.SubElement(root, tag).text = text
                    src = ET.SubElement(root, "source")
                ET.SubElement(src, "database").text = "Unknown"
                size = ET.SubElement(root, "size")
                for tag, val in (("width", aug_img.shape[1]), ("height", aug_img.shape[0]), ("depth", aug_img.shape[2])):
                    ET.SubElement(size, tag).text = str(val)
                ET.SubElement(root, "segmented").text = "0"
                for box, lbl in zip(aug_boxes, aug_labels):
                    obj = ET.SubElement(root, "object")
                    ET.SubElement(obj, "name").text = lbl
                    for tag in ("pose", "Unspecified"), ("truncated", "0"), ("difficult", "0"):
                        ET.SubElement(obj, tag[0]).text = tag[1]
                    bb = ET.SubElement(obj, "bndbox")
                    for tag, val in zip(("xmin", "ymin", "xmax", "ymax"), map(int, map(round, box))):
                        ET.SubElement(bb, tag).text = str(val)
                ET.ElementTree(root).write(xml_out_path, encoding="utf-8", xml_declaration=True)
            except Exception as e:
                log(f"Save XML error {new_name}: {e}")
                if os.path.exists(img_out_path):
                    os.remove(img_out_path)
                continue

            # counts
            for lbl in aug_labels:
                curr_counts[lbl] += 1
            total_created += 1

    # 5. final stats ---------------------------------------------------------
    final_counts, _, _ = analyse(xml_out_dir)
    log("\n" + "=" * 70)
    log("AUGMENTATION COMPLETE")
    log("=" * 70)
    log("Class            Init   Final   Target  Status")
    log("-" * 50)
    achieved = 0
    for c in all_classes:
        ini, fin, tgt = init_counts[c], final_counts[c], target_per_class
        status = "✓" if fin >= tgt else "✗"
        if fin >= tgt:
            achieved += 1
        log(f"{c:17}{ini:6}{fin:8}{tgt:8}  {status}")
    log(f"\nTotal created: {total_created}")
    log(f"Targets hit:   {achieved}/{len(all_classes)}")
    log_f.close()
    return total_created, final_counts


# ------------------------------------------------------------------
# ---------------  convenience wrapper  ----------------------------
# ------------------------------------------------------------------
def run_augmentation(img_input, xml_input, output_base, target=2000):
    """Full pipeline: copy originals + augment."""
    print("\n" + "=" * 70)
    print("DATASET AUGMENTATION PIPELINE – COMPLETE")
    print("=" * 70)
    os.makedirs(output_base, exist_ok=True)

    img_out = os.path.join(output_base, "images")
    xml_out = os.path.join(output_base, "annotations")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(xml_out, exist_ok=True)

    # copy / symlink originals
    def mirror(src_dir, dst_dir, exts):
        for f in os.listdir(src_dir):
            if any(f.lower().endswith(e) for e in exts):
                src = os.path.join(src_dir, f)
                dst = os.path.join(dst_dir, f)
                if not os.path.exists(dst):
                    if os.name != "nt":
                        os.symlink(src, dst)
                    else:
                        shutil.copy2(src, dst)

    mirror(img_input, img_out, [".jpg", ".jpeg", ".png"])
    mirror(xml_input, xml_out, [".xml"])

    # augment
    run_fixed_augmentation(img_out, xml_out,
                           target_per_class=target,
                           output_dir=None,  # write into same folder
                           report_file=os.path.join(output_base, "augmentation_report.txt"))
    print("\nDone – output:", output_base)
    return output_base


# ------------------------------------------------------------------
if __name__ == "1__main__":
    TRAIN_IMG = "/kaggle/working/final_dataset/train/images"
    TRAIN_XML = "/kaggle/working/final_dataset/train/annotations"
    OUT_DIR   = "/kaggle/working/final_dataset/train_aug"

    run_augmentation(TRAIN_IMG, TRAIN_XML, OUT_DIR, target=9000)

# Reading Dataset

In [5]:
import os
import xml.etree.ElementTree as ET

def build_class_map(xml_dir):
    class_names = set()

    for f in os.listdir(xml_dir):
        if f.endswith(".xml"):
            tree = ET.parse(os.path.join(xml_dir, f))
            root = tree.getroot()
            for obj in root.findall("object"):
                class_names.add(obj.find("name").text.strip())

    class_names = sorted(list(class_names))
    class_map = {"__background__": 0}

    for i, name in enumerate(class_names, start=1):
        class_map[name] = i

    return class_map

def parse_voc(xml_path, class_map):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    boxes, labels = [], []

    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name not in class_map:
            continue

        b = obj.find("bndbox")
        boxes.append([
            float(b.find("xmin").text),
            float(b.find("ymin").text),
            float(b.find("xmax").text),
            float(b.find("ymax").text),
        ])
        labels.append(class_map[name])

    if len(boxes) == 0:
        return None

    return {
        "boxes": torch.tensor(boxes, dtype=torch.float32),
        "labels": torch.tensor(labels, dtype=torch.int64)
    }

def get_image_classes(xml_path, class_map):
    """Get all classes present in an image"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    classes = set()
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name in class_map:
            classes.add(name)
    
    return list(classes)

def get_object_statistics(xml_dir, class_map):
    """Get detailed object statistics from XML files"""
    total_objects = 0
    images_with_objects = 0
    images_without_objects = 0
    class_object_counts = defaultdict(int)
    images_per_class = defaultdict(int)
    objects_per_image = []
    
    xml_files = [f for f in os.listdir(xml_dir) if f.endswith(".xml")]
    
    for xml_file in xml_files:
        xml_path = os.path.join(xml_dir, xml_file)
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        objects_in_image = 0
        classes_in_image = set()
        
        for obj in root.findall("object"):
            name = obj.find("name").text.strip()
            if name in class_map:
                total_objects += 1
                objects_in_image += 1
                class_object_counts[name] += 1
                classes_in_image.add(name)
        
        objects_per_image.append(objects_in_image)
        
        if objects_in_image > 0:
            images_with_objects += 1
            for cls in classes_in_image:
                images_per_class[cls] += 1
        else:
            images_without_objects += 1
    
    # Calculate statistics
    avg_objects_per_image = total_objects / len(xml_files) if xml_files else 0
    max_objects_per_image = max(objects_per_image) if objects_per_image else 0
    min_objects_per_image = min(objects_per_image) if objects_per_image else 0
    
    return {
        'total_objects': total_objects,
        'total_images': len(xml_files),
        'images_with_objects': images_with_objects,
        'images_without_objects': images_without_objects,
        'avg_objects_per_image': avg_objects_per_image,
        'max_objects_per_image': max_objects_per_image,
        'min_objects_per_image': min_objects_per_image,
        'class_object_counts': dict(class_object_counts),
        'images_per_class': dict(images_per_class),
        'objects_per_image_list': objects_per_image
    }

# Train Test Val Split

In [6]:
def create_stratified_splits(xml_dir, class_map, train_ratio=0.7, val_ratio=0.15, random_seed=42):
    """Create stratified splits using a simple approach"""
    # Get all XML files
    all_xml_files = sorted([f for f in os.listdir(xml_dir) if f.endswith(".xml")])
    
    # Create a simple stratification based on the presence of each class
    strat_labels = []
    
    for xml_file in all_xml_files:
        xml_path = os.path.join(xml_dir, xml_file)
        classes = get_image_classes(xml_path, class_map)
        
        if not classes:
            # Empty image
            strat_labels.append("empty")
        else:
            # Use the first class as stratification label
            strat_labels.append(classes[0])
    
    # Count occurrences of each stratification label
    from collections import Counter
    label_counts = Counter(strat_labels)
    
    # Separate files by their stratification label
    label_to_files = defaultdict(list)
    for xml_file, label in zip(all_xml_files, strat_labels):
        label_to_files[label].append(xml_file)
    
    # Initialize splits
    train_files = []
    val_files = []
    test_files = []
    
    # For each label, split its files proportionally
    for label, files in label_to_files.items():
        n_files = len(files)
        
        # Calculate number of files for each split
        n_train = max(1, int(n_files * train_ratio))
        n_val = max(1, int(n_files * val_ratio))
        n_test = n_files - n_train - n_val
        
        # Adjust if n_test is negative
        if n_test < 0:
            n_train = max(1, n_train + n_test)
            n_test = 0
        
        # Shuffle files
        random.Random(random_seed).shuffle(files)
        
        # Split
        train_files.extend(files[:n_train])
        val_files.extend(files[n_train:n_train + n_val])
        test_files.extend(files[n_train + n_val:])
    
    # Final shuffle
    random.Random(random_seed).shuffle(train_files)
    random.Random(random_seed + 1).shuffle(val_files)
    random.Random(random_seed + 2).shuffle(test_files)
    
    return train_files, val_files, test_files

def get_split_statistics(xml_dir, class_map, train_files, val_files, test_files):
    """Get detailed statistics for each split"""
    
    def get_split_stats(files):
        total_objects = 0
        class_counts = defaultdict(int)
        images_with_objects = 0
        
        for xml_file in files:
            xml_path = os.path.join(xml_dir, xml_file)
            tree = ET.parse(xml_path)
            root = tree.getroot()
            
            objects_in_image = 0
            for obj in root.findall("object"):
                name = obj.find("name").text.strip()
                if name in class_map:
                    total_objects += 1
                    objects_in_image += 1
                    class_counts[name] += 1
            
            if objects_in_image > 0:
                images_with_objects += 1
        
        return {
            'total_images': len(files),
            'total_objects': total_objects,
            'images_with_objects': images_with_objects,
            'images_without_objects': len(files) - images_with_objects,
            'avg_objects_per_image': total_objects / len(files) if files else 0,
            'class_counts': dict(class_counts)
        }
    
    train_stats = get_split_stats(train_files)
    val_stats = get_split_stats(val_files)
    test_stats = get_split_stats(test_files)
    
    return train_stats, val_stats, test_stats

# Dataset Path, intial analysis & split ratio

In [7]:
# ================== MAIN TRAINING LOOP ==================
# paths
IMAGE_DIR = "/kaggle/input/datasets/carunmanikandan/pv-multi-defect-main/PV-Multi-Defect-main/JPEGImages"
XML_DIR   = "/kaggle/input/datasets/carunmanikandan/pv-multi-defect-main/PV-Multi-Defect-main/Annotations"

# build classes
CLASS_MAP = build_class_map(XML_DIR)
NUM_CLASSES = len(CLASS_MAP)

# Get detailed object statistics
logging.info("\n" + "=" * 80)
logging.info("DATASET STATISTICS")
logging.info("=" * 80)

# Get overall statistics
stats = get_object_statistics(XML_DIR, CLASS_MAP)

# Log overall statistics table
overall_stats_table = [
    ["Total Images", stats['total_images']],
    ["Images with Objects", stats['images_with_objects']],
    ["Images without Objects", stats['images_without_objects']],
    ["Total Objects", stats['total_objects']],
    ["Avg Objects per Image", f"{stats['avg_objects_per_image']:.2f}"],
    ["Max Objects per Image", stats['max_objects_per_image']],
    ["Min Objects per Image", stats['min_objects_per_image']],
    ["Images without Objects %", f"{(stats['images_without_objects']/stats['total_images']*100):.1f}%"]
]

logging.info("\nOVERALL DATASET STATISTICS:")
overall_table = tabulate(overall_stats_table, tablefmt="grid")
for line in overall_table.split('\n'):
    logging.info(line)

# Log class-wise object statistics
class_stats_table = []
for cls, count in stats['class_object_counts'].items():
    class_stats_table.append([
        cls,
        count,
        f"{(count/stats['total_objects']*100):.1f}%",
        stats['images_per_class'][cls],
        f"{(stats['images_per_class'][cls]/stats['total_images']*100):.1f}%"
    ])

logging.info("\nCLASS-WISE OBJECT STATISTICS:")
class_headers = ["Class", "Object Count", "% of Total", "Images with Class", "% of Images"]
class_table = tabulate(class_stats_table, headers=class_headers, tablefmt="grid")
for line in class_table.split('\n'):
    logging.info(line)

logging.info("=" * 80)

# Create stratified splits
logging.info("\nCreating stratified splits (70-15-15)...")
train_files, val_files, test_files = create_stratified_splits(
    XML_DIR, CLASS_MAP, 
    train_ratio=0.7, 
    val_ratio=0.15, 
    random_seed=42
)

2026-02-25 09:50:40,203 | INFO | 
2026-02-25 09:50:40,205 | INFO | DATASET STATISTICS
2026-02-25 09:50:40,206 | INFO | ================================================================================
2026-02-25 09:50:40,942 | INFO | 
OVERALL DATASET STATISTICS:
2026-02-25 09:50:40,944 | INFO | +--------------------------+------+
2026-02-25 09:50:40,945 | INFO | | Total Images             | 1106 |
2026-02-25 09:50:40,946 | INFO | +--------------------------+------+
2026-02-25 09:50:40,947 | INFO | | Images with Objects      | 1105 |
2026-02-25 09:50:40,948 | INFO | +--------------------------+------+
2026-02-25 09:50:40,948 | INFO | | Images without Objects   | 1    |
2026-02-25 09:50:40,949 | INFO | +--------------------------+------+
2026-02-25 09:50:40,949 | INFO | | Total Objects            | 3981 |
2026-02-25 09:50:40,950 | INFO | +--------------------------+------+
2026-02-25 09:50:40,950 | INFO | | Avg Objects per Image    | 3.60 |
2026-02-25 09:50:40,951 | INFO | +--------------

# Saving the intial dataset after spliting

In [8]:
# Get split statistics
train_stats, val_stats, test_stats = get_split_statistics(
    XML_DIR, CLASS_MAP, train_files, val_files, test_files
)

# Log split statistics in a comprehensive table
logging.info("\n" + "=" * 80)
logging.info("TRAIN-VAL-TEST SPLIT STATISTICS")
logging.info("=" * 80)

# Create comprehensive split statistics table
split_stats_table = []

# Headers
headers = ["Metric", "Train", "Validation", "Test", "Total"]

# Add rows for each metric
split_stats_table.append(["Images", 
                         f"{train_stats['total_images']} ({train_stats['total_images']/(train_stats['total_images']+val_stats['total_images']+test_stats['total_images'])*100:.1f}%)",
                         f"{val_stats['total_images']} ({val_stats['total_images']/(train_stats['total_images']+val_stats['total_images']+test_stats['total_images'])*100:.1f}%)",
                         f"{test_stats['total_images']} ({test_stats['total_images']/(train_stats['total_images']+val_stats['total_images']+test_stats['total_images'])*100:.1f}%)",
                         f"{train_stats['total_images'] + val_stats['total_images'] + test_stats['total_images']}"])

split_stats_table.append(["Objects",
                         f"{train_stats['total_objects']} ({train_stats['total_objects']/(train_stats['total_objects']+val_stats['total_objects']+test_stats['total_objects'])*100:.1f}%)",
                         f"{val_stats['total_objects']} ({val_stats['total_objects']/(train_stats['total_objects']+val_stats['total_objects']+test_stats['total_objects'])*100:.1f}%)",
                         f"{test_stats['total_objects']} ({test_stats['total_objects']/(train_stats['total_objects']+val_stats['total_objects']+test_stats['total_objects'])*100:.1f}%)",
                         f"{train_stats['total_objects'] + val_stats['total_objects'] + test_stats['total_objects']}"])

split_stats_table.append(["Avg Objects/Image",
                         f"{train_stats['avg_objects_per_image']:.2f}",
                         f"{val_stats['avg_objects_per_image']:.2f}",
                         f"{test_stats['avg_objects_per_image']:.2f}",
                         f"{stats['avg_objects_per_image']:.2f}"])

split_stats_table.append(["Images with Objects",
                         f"{train_stats['images_with_objects']} ({train_stats['images_with_objects']/train_stats['total_images']*100:.1f}%)",
                         f"{val_stats['images_with_objects']} ({val_stats['images_with_objects']/val_stats['total_images']*100:.1f}%)",
                         f"{test_stats['images_with_objects']} ({test_stats['images_with_objects']/test_stats['total_images']*100:.1f}%)",
                         f"{stats['images_with_objects']} ({stats['images_with_objects']/stats['total_images']*100:.1f}%)"])

split_stats_table.append(["Images without Objects",
                         f"{train_stats['images_without_objects']} ({train_stats['images_without_objects']/train_stats['total_images']*100:.1f}%)",
                         f"{val_stats['images_without_objects']} ({val_stats['images_without_objects']/val_stats['total_images']*100:.1f}%)",
                         f"{test_stats['images_without_objects']} ({test_stats['images_without_objects']/test_stats['total_images']*100:.1f}%)",
                         f"{stats['images_without_objects']} ({stats['images_without_objects']/stats['total_images']*100:.1f}%)"])

# Log the comprehensive split table
split_table = tabulate(split_stats_table, headers=headers, tablefmt="grid")
for line in split_table.split('\n'):
    logging.info(line)

# Log class distribution in each split
logging.info("\nCLASS DISTRIBUTION ACROSS SPLITS:")

# Get all unique classes (excluding background)
all_classes = sorted([cls for cls in CLASS_MAP.keys() if cls != "__background__"])

class_dist_table = []
for cls in all_classes:
    train_count = train_stats['class_counts'].get(cls, 0)
    val_count = val_stats['class_counts'].get(cls, 0)
    test_count = test_stats['class_counts'].get(cls, 0)
    total_count = stats['class_object_counts'].get(cls, 0)
    
    class_dist_table.append([
        cls,
        f"{train_count} ({train_count/total_count*100:.1f}%)" if total_count > 0 else "0",
        f"{val_count} ({val_count/total_count*100:.1f}%)" if total_count > 0 else "0",
        f"{test_count} ({test_count/total_count*100:.1f}%)" if total_count > 0 else "0",
        total_count
    ])

class_dist_headers = ["Class", "Train Objects", "Val Objects", "Test Objects", "Total Objects"]
class_dist_display = tabulate(class_dist_table, headers=class_dist_headers, tablefmt="grid")
for line in class_dist_display.split('\n'):
    logging.info(line)

logging.info("=" * 80)

2026-02-25 09:50:42,677 | INFO | 
2026-02-25 09:50:42,678 | INFO | TRAIN-VAL-TEST SPLIT STATISTICS
2026-02-25 09:50:42,679 | INFO | ================================================================================
2026-02-25 09:50:42,681 | INFO | +------------------------+--------------+--------------+--------------+--------------+
2026-02-25 09:50:42,681 | INFO | | Metric                 | Train        | Validation   | Test         | Total        |
2026-02-25 09:50:42,682 | INFO | +========================+==============+==============+==============+==============+
2026-02-25 09:50:42,683 | INFO | | Images                 | 772 (69.8%)  | 164 (14.8%)  | 170 (15.4%)  | 1106         |
2026-02-25 09:50:42,683 | INFO | +------------------------+--------------+--------------+--------------+--------------+
2026-02-25 09:50:42,684 | INFO | | Objects                | 2678 (67.3%) | 566 (14.2%)  | 737 (18.5%)  | 3981         |
2026-02-25 09:50:42,685 | INFO | +------------------------+--------

In [9]:
import shutil
import os

def save_organized_dataset(file_list, source_img_dir, source_xml_dir, 
                           base_target_dir, split_name, 
                           img_ext=".jpg", debug=False):
    """
    Saves images and XMLs into:
    base_target_dir/split_name/images/
    base_target_dir/split_name/annotations/

    Debug logs are shown only if debug=True.
    """

    if debug:
        print("\n" + "="*60)
        print(f"[INFO] Processing split: {split_name}")
        print(f"[INFO] Total files received: {len(file_list)}")

    os.makedirs(base_target_dir, exist_ok=True)

    images_path = os.path.join(base_target_dir, split_name, "images")
    annotations_path = os.path.join(base_target_dir, split_name, "annotations")

    os.makedirs(images_path, exist_ok=True)
    os.makedirs(annotations_path, exist_ok=True)

    if debug:
        print(f"[DEBUG] Images path: {images_path}")
        print(f"[DEBUG] Annotations path: {annotations_path}")

    copied_count = 0
    missing_images = 0
    missing_xmls = 0

    for idx, file_name in enumerate(file_list):
        base_name = os.path.splitext(file_name)[0]

        img_src = os.path.join(source_img_dir, f"{base_name}{img_ext}")
        xml_src = os.path.join(source_xml_dir, f"{base_name}.xml")

        if debug:
            print(f"[DEBUG] ({idx+1}/{len(file_list)}) Processing: {base_name}")

        if not os.path.exists(img_src):
            missing_images += 1
            if debug:
                print(f"[WARNING] Missing image: {img_src}")
            continue

        if not os.path.exists(xml_src):
            missing_xmls += 1
            if debug:
                print(f"[WARNING] Missing XML: {xml_src}")
            continue

        try:
            shutil.copy(img_src, images_path)
            shutil.copy(xml_src, annotations_path)
            copied_count += 1
            if debug:
                print(f"[SUCCESS] Copied: {base_name}")
        except Exception as e:
            if debug:
                print(f"[ERROR] Failed to copy {base_name}: {e}")

    # Always show final summary (important info)
    print(f"[{split_name.upper()}] Saved {copied_count} pairs")
    
    if debug:
        print(f"Missing images: {missing_images}")
        print(f"Missing XMLs  : {missing_xmls}")
        print("="*60)

In [10]:
# --- Execute for all splits ---
save_organized_dataset(train_files, IMAGE_DIR, XML_DIR, "/kaggle/working/final_dataset", "train")
save_organized_dataset(val_files, IMAGE_DIR, XML_DIR, "/kaggle/working/final_dataset", "val")
save_organized_dataset(test_files, IMAGE_DIR, XML_DIR, "/kaggle/working/final_dataset", "test")

[TRAIN] Saved 772 pairs
[VAL] Saved 164 pairs
[TEST] Saved 170 pairs


In [11]:
run_augmentation('/kaggle/working/final_dataset/train/images', '/kaggle/working/final_dataset/train/annotations', "/kaggle/working/final_dataset/train_aug")


DATASET AUGMENTATION PIPELINE – COMPLETE
[2026-02-25 09:50:53] ======================================================================
[2026-02-25 09:50:53] FIXED AUGMENTATION – COMPLETE DIVERSE PIPELINE
[2026-02-25 09:50:53] ======================================================================
[2026-02-25 09:50:53] 
Initial counts:
[2026-02-25 09:50:53]   black_border: 181
[2026-02-25 09:50:53]   broken: 62
[2026-02-25 09:50:53]   hot_spot: 1385
[2026-02-25 09:50:53]   no_electricity: 129
[2026-02-25 09:50:53]   scratch: 921
[2026-02-25 09:50:53] 
Augmentation needs:
[2026-02-25 09:50:53]   black_border: 1819
[2026-02-25 09:50:53]   broken: 1938
[2026-02-25 09:50:53]   hot_spot: 615
[2026-02-25 09:50:53]   no_electricity: 1871
[2026-02-25 09:50:53]   scratch: 1079
[2026-02-25 09:50:53] 
Creating schedule...
[2026-02-25 09:50:53] Planned augmentations: 613
[2026-02-25 09:50:53] 
Starting augmentation...


Augmenting:  52%|█████▏    | 320/613 [00:54<00:50,  5.84it/s]


[2026-02-25 09:51:48] Global target reached – stopping early.
[2026-02-25 09:51:48] 
[2026-02-25 09:51:48] AUGMENTATION COMPLETE
[2026-02-25 09:51:48] ======================================================================
[2026-02-25 09:51:48] Class            Init   Final   Target  Status
[2026-02-25 09:51:48] --------------------------------------------------
[2026-02-25 09:51:48] black_border        181    2071    2000  ✓
[2026-02-25 09:51:48] broken               62    2025    2000  ✓
[2026-02-25 09:51:48] hot_spot           1385    7014    2000  ✓
[2026-02-25 09:51:48] no_electricity      129    2000    2000  ✓
[2026-02-25 09:51:48] scratch             921    2001    2000  ✓
[2026-02-25 09:51:48] 
Total created: 4695
[2026-02-25 09:51:48] Targets hit:   5/5

Done – output: /kaggle/working/final_dataset/train_aug


'/kaggle/working/final_dataset/train_aug'

# verify the augmentation

In [12]:
import os
import cv2
import matplotlib.pyplot as plt
from lxml import etree

def verify_saved_augmentation(img_dir, xml_dir, num_samples=10, output_filename="augmentation_verification.png"):
    # Find all augmented files
    aug_xmls = [f for f in os.listdir(xml_dir) if "_aug" in f]
    
    if not aug_xmls:
        print("No augmented files found!")
        return

    num_to_show = min(num_samples, len(aug_xmls))
    cols = 5
    rows = (num_to_show // cols) + (1 if num_to_show % cols != 0 else 0)
    
    # Increased figsize height multiplier to 6 for better clarity with 100 samples
    fig, axes = plt.subplots(rows, cols, figsize=(25, 6 * rows))
    
    if num_to_show == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i in range(num_to_show):
        sample_xml = aug_xmls[i]
        # Check for multiple possible extensions
        base_name = sample_xml.replace(".xml", "")
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".JPG"]:
            temp_path = os.path.join(img_dir, base_name + ext)
            if os.path.exists(temp_path):
                img_path = temp_path
                break
        
        img = cv2.imread(img_path) if img_path else None
        
        if img is None:
            axes[i].text(0.5, 0.5, f"Missing Image:\n{base_name}", 
                         ha='center', va='center', color='red')
            axes[i].axis('off')
            continue
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Parse XML
        try:
            tree = etree.parse(os.path.join(xml_dir, sample_xml))
            for obj in tree.getroot().findall("object"):
                obj_name = obj.find("name").text
                bbox = obj.find("bndbox")
                xmin = int(bbox.find("xmin").text)
                ymin = int(bbox.find("ymin").text)
                xmax = int(bbox.find("xmax").text)
                ymax = int(bbox.find("ymax").text)
                
                # Draw Rectangle and Label
                cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (255, 0, 0), 8)
                cv2.putText(img, obj_name, (xmin, ymin - 15), 
                            cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255, 0, 0), 5)
        except Exception as e:
            print(f"Error parsing {sample_xml}: {e}")
        
        axes[i].imshow(img)
        axes[i].set_title(sample_xml, fontsize=12)
        axes[i].axis('off')

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    
    # Save the file
    plt.savefig(output_filename, dpi=100, bbox_inches='tight')
    print(f"Verification grid saved as: {output_filename}")
    plt.close() # Close figure to free up memory

# Usage:

In [13]:
verify_saved_augmentation('/kaggle/working/final_dataset/train_aug/images', '/kaggle/working/final_dataset/train_aug/annotations',num_samples=100)


Verification grid saved as: augmentation_verification.png


# pre dataloader

In [14]:
import cv2
from torch.utils.data import Dataset

class FasterRCNNDataset(Dataset):
    def __init__(self, image_dir, xml_dir, class_map, split='train', 
                 train_files=None, val_files=None, test_files=None):
        """
        split: 'train', 'val', or 'test'
        train_files, val_files, test_files: pre-computed splits
        """
        self.image_dir = image_dir
        self.xml_dir = xml_dir
        self.class_map = class_map
        
        # Assign files based on split
        if split == 'train':
            self.xml_files = train_files
        elif split == 'val':
            self.xml_files = val_files
        elif split == 'test':
            self.xml_files = test_files
        else:
            raise ValueError("split must be 'train', 'val', or 'test'")
        
        logging.info(f"{split.upper()} dataset: {len(self.xml_files)} samples")

    def __len__(self):
        return len(self.xml_files)

    def __getitem__(self, idx):
        xml_path = os.path.join(self.xml_dir, self.xml_files[idx])
        target = parse_voc(xml_path, self.class_map)
    
        if target is None:
            return None
    
        tree = ET.parse(xml_path)
        root = tree.getroot()
        filename = root.find("filename").text
    
        img = cv2.imread(os.path.join(self.image_dir, filename))
        if img is None:
            logging.warning(f"Image not found: {filename}")
            return None
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
        return img, target

In [15]:
import os
import xml.etree.ElementTree as ET

def get_valid_train_files(xml_dir, img_dir, class_map):
    """
    Identifies XML/Image pairs that exist AND contain at least 
    one object from the provided class_map.
    """
    all_xml_files = sorted([f for f in os.listdir(xml_dir) if f.endswith(".xml")])
    train_files = []
    
    # Target classes from your map keys or values
    target_classes = set(class_map.keys())

    for xml_name in all_xml_files:
        base_name = os.path.splitext(xml_name)[0]
        img_name = base_name + ".jpg"
        
        src_xml = os.path.join(xml_dir, xml_name)
        src_img = os.path.join(img_dir, img_name)
        
        # 1. Check if image exists
        if os.path.exists(src_img):
            # 2. Parse XML to check for valid classes
            try:
                tree = ET.parse(src_xml)
                root = tree.getroot()
                
                # Check if any object in the XML is in our class_map
                has_valid_object = False
                for obj in root.findall('object'):
                    obj_name = obj.find('name').text
                    if obj_name in target_classes:
                        has_valid_object = True
                        break
                
                if has_valid_object:
                    train_files.append(xml_name)
            except Exception as e:
                print(f"Error reading {xml_name}: {e}")
                
    print(f"Verified {len(train_files)} files containing target classes.")
    return train_files

In [16]:
 train_files =get_valid_train_files('final_dataset/train_aug/annotations/', 'final_dataset/train_aug/images/',  CLASS_MAP)

Verified 5466 files containing target classes.


# Detection matrix

In [17]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return [], []
    return tuple(zip(*batch))

from torchmetrics.detection.mean_ap import MeanAveragePrecision

def box_iou(box1, box2):
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])

    lt = torch.max(box1[:, None, :2], box2[:, :2])
    rb = torch.min(box1[:, None, 2:], box2[:, 2:])

    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area1[:, None] + area2 - inter

    return inter / union

def detection_metrics(
    pred_boxes,
    pred_labels,
    gt_boxes,
    gt_labels,
    iou_threshold=0.5
):
    """
    pred_boxes: Tensor[N, 4]
    pred_labels: Tensor[N]
    gt_boxes:   Tensor[M, 4]
    gt_labels: Tensor[M]
    """

    if len(pred_boxes) == 0:
        return 0.0, 0.0, 0.0

    if len(gt_boxes) == 0:
        return 0.0, 0.0, 0.0

    ious = box_iou(pred_boxes, gt_boxes)
    matched_gt = set()

    TP, FP = 0, 0

    for i in range(len(pred_boxes)):
        best_iou, gt_idx = ious[i].max(0)

        if best_iou >= iou_threshold and gt_idx.item() not in matched_gt:
            if pred_labels[i] == gt_labels[gt_idx]:
                TP += 1
                matched_gt.add(gt_idx.item())
            else:
                FP += 1
        else:
            FP += 1

    FN = len(gt_boxes) - len(matched_gt)

    precision = TP / (TP + FP + 1e-8)
    recall    = TP / (TP + FN + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1

# some class defination

In [18]:
ALL_CLASS_NAMES = list(CLASS_MAP.keys())
BACKGROUND_CLASS_NAME = '__background__'
BACKGROUND_CLASS_ID = CLASS_MAP[BACKGROUND_CLASS_NAME]

NON_BACKGROUND_CLASS_NAMES = [
    c for c in ALL_CLASS_NAMES if c != BACKGROUND_CLASS_NAME
]

# Training Configuration

In [19]:
class TrainingConfig:
    def __init__(self):
        self.output_root = ""
        self.num_epochs = 200
        self.learning_rate = 1e-4
        self.weight_decay = 1e-4
        self.confidence_thresholds = [0.0]
        self.batch_size = 16
        self.final_path=""
        self.create_directories()

    def create_directories(self):
        subdirs = [
            "metrics",
            "models",
        ]
        for d in subdirs:
            os.makedirs(os.path.join(self.output_root, d), exist_ok=True)

config = TrainingConfig()

In [20]:
# Create datasets
train_dataset = FasterRCNNDataset(
    'final_dataset/train_aug/images/', "final_dataset/train_aug/annotations/", CLASS_MAP,
    split='train', 
    train_files=train_files,
    val_files=val_files,
    test_files=test_files
)

val_dataset = FasterRCNNDataset(
    IMAGE_DIR, XML_DIR, CLASS_MAP,
    split='val',
    train_files=train_files,
    val_files=val_files,
    test_files=test_files
)

test_dataset = FasterRCNNDataset(
    IMAGE_DIR, XML_DIR, CLASS_MAP,
    split='test',
    train_files=train_files,
    val_files=val_files,
    test_files=test_files
)

2026-02-25 09:52:06,292 | INFO | TRAIN dataset: 5466 samples
2026-02-25 09:52:06,293 | INFO | VAL dataset: 164 samples
2026-02-25 09:52:06,294 | INFO | TEST dataset: 170 samples


In [21]:
# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

# CSV init

In [22]:
# 1. Main training metrics CSV
csv_file = os.path.join(config.output_root, "metrics/training_metrics.csv")
with open(csv_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'epoch', 'train_loss',
        'val_precision', 'val_recall', 'val_f1',
        'val_map_05', 'best_epoch'
    ])
# 2. Class-wise mAP@0.5 CSV
class_map_csv_file = os.path.join(config.output_root, "metrics/classwise_map_05.csv")
with open(class_map_csv_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'class_name', 'class_id', 'map_05', 'weight'])

# 3. Per-class precision, recall, F1 CSV

# model Archhicture

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import FeaturePyramidNetwork, MultiScaleRoIAlign

# =========================================================
# Basic Conv
# =========================================================
class Conv(nn.Module):
    def __init__(self, c1, c2, k=1, s=1, p=None, g=1):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, p or k // 2, groups=g, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class DetailConvBlock(nn.Module):
    """
    High-resolution detail extractor for scratches / cracks.
    """
    def __init__(self, ch):
        super().__init__()
        self.dw7 = nn.Conv2d(ch, ch, 7, padding=3, groups=ch, bias=False)
        self.dw9 = nn.Conv2d(ch, ch, 9, padding=4, groups=ch, bias=False)
        self.dw11 = nn.Conv2d(ch, ch, 11, padding=5, groups=ch, bias=False)
        self.dw13 = nn.Conv2d(ch, ch, 13, padding=6, groups=ch, bias=False)
        self.dw_h = nn.Conv2d(ch, ch, (1, 9), padding=(0, 4), groups=ch, bias=False)
        self.dw_v = nn.Conv2d(ch, ch, (9, 1), padding=(4, 0), groups=ch, bias=False)

        # Fuse: 6 branches * ch -> ch
        self.fuse = nn.Sequential(
            nn.Conv2d(ch * 6, ch, 1, bias=False),
            nn.BatchNorm2d(ch),
            nn.SiLU()
        )

    def forward(self, x):
        f7 = self.dw7(x)
        f9 = self.dw9(x)
        f11 = self.dw11(x)
        f13 = self.dw13(x)
        fh = self.dw_h(x)
        fv = self.dw_v(x)
        out = torch.cat([f7, f9, f11, f13, fh, fv], dim=1)
        return x + self.fuse(out)

class CoordAtt(nn.Module):
    def __init__(self, inp, reduction=32):
        super().__init__()
        mip = max(8, inp // reduction)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU()
        self.conv_h = nn.Conv2d(mip, inp, kernel_size=1, bias=False)
        self.conv_w = nn.Conv2d(mip, inp, kernel_size=1, bias=False)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()
        return identity * a_h * a_w

class SqueezeExcitation(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.SiLU(),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.SiLU()
        self.fc2 = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return x * self.sigmoid(out)

class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c_ * 4, c2, 1, 1)
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        return self.cv2(torch.cat([x, y1, y2, y3], 1))

class C2f(nn.Module):
    def __init__(self, c1, c2, n=2):
        super().__init__()
        self.cv1 = Conv(c1, c2, 1, 1)
        self.cv2 = Conv(c1, c2, 1, 1)
        self.cv3 = Conv((n + 2) * c2, c2, 1)
        self.m = nn.ModuleList([Conv(c2, c2, 3, 1) for _ in range(n)])

    def forward(self, x):
        y = [self.cv1(x), self.cv2(x)]
        for m in self.m:
            y.append(m(y[-1]))
        return self.cv3(torch.cat(y, 1))

class FastViTBackboneWithFPN(nn.Module):
    def __init__(self, out_channels=256, freeze_backbone=True):
        super().__init__()
        self.backbone = timm.create_model(
            "fastvit_t8",
            pretrained=True,
            features_only=True,
            out_indices=(0, 1, 2, 3)
        )
        in_channels = self.backbone.feature_info.channels()
        print("FastViT channels:", in_channels)

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=in_channels,
            out_channels=out_channels
        )

        self.c2f = nn.ModuleList([C2f(out_channels, out_channels, n=2) for _ in range(4)])
        self.ca = nn.ModuleList([CoordAtt(out_channels) for _ in range(4)])
        self.detail = nn.ModuleList([DetailConvBlock(out_channels) for _ in range(2)])
        self.sppf = SPPF(out_channels, out_channels)
        self.se_blocks = nn.ModuleList([SqueezeExcitation(out_channels) for _ in range(4)])
        self.cbam_channel = nn.ModuleList([ChannelAttention(out_channels) for _ in range(4)])
        
        self.bu_downsample_convs = nn.ModuleList([Conv(out_channels, out_channels, 3, 2) for _ in range(3)])
        self.bu_fusion_convs = nn.ModuleList([Conv(out_channels * 2, out_channels, 1, 1) for _ in range(3)])

        self.out_channels = out_channels
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        feats = self.backbone(x)
        feats = {str(i): f for i, f in enumerate(feats)}
        fpn_feats = self.fpn(feats)
        outputs = {}

        for i, (k, v) in enumerate(fpn_feats.items()):
            v = self.c2f[i](v)
            v = self.ca[i](v)
            v = self.se_blocks[i](v)
            v = self.cbam_channel[i](v)
            if i < 2:
                v = self.detail[i](v)
            if i == len(fpn_feats) - 1:
                v = self.sppf(v)
            outputs[k] = v

        bu_features = [outputs['0'], outputs['1'], outputs['2'], outputs['3']]
        for i in range(1, len(bu_features)):
            prev_level_feature = bu_features[i-1]
            downsampled_prev = self.bu_downsample_convs[i-1](prev_level_feature)
            
            # Interpolation check to prevent crashes
            target_shape = bu_features[i].shape[2:]
            if downsampled_prev.shape[2:] != target_shape:
                downsampled_prev = F.interpolate(downsampled_prev, size=target_shape, mode='nearest')

            fused_feature = torch.cat([bu_features[i], downsampled_prev], dim=1)
            bu_features[i] = self.bu_fusion_convs[i-1](fused_feature)

        outputs['0'], outputs['1'], outputs['2'], outputs['3'] = bu_features[0], bu_features[1], bu_features[2], bu_features[3]
        return outputs
# =========================================================
# Stage-wise Freezing
# =========================================================
def freeze_all_backbone(model):
    for p in model.backbone.backbone.parameters():
        p.requires_grad = False

# =========================================================
# Model Builder
# =========================================================

def build_model(num_classes):
    backbone = FastViTBackboneWithFPN(out_channels=256)
    
    # --- FIX START ---
    custom_aspect_ratios = ((0.11, 0.24, 0.45, 0.99, 2.54),) * 4 # Changed from 5 to 4
    custom_sizes = ((33,), (55,), (80,), (160,),(330,))

    anchor_generator = AnchorGenerator(
        # 4. Correct Argument Assignment (Sizes -> sizes, Ratios -> aspect_ratios)
        sizes=custom_sizes, 
        aspect_ratios=custom_aspect_ratios
    )
    # --- FIX END ---

    roi_pooler = MultiScaleRoIAlign(
        featmap_names=["0", "1", "2", "3"],
        output_size=7,
        sampling_ratio=2
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
        min_size=512,
        max_size=1024
    )
    freeze_all_backbone(model)
    return model



def unfreeze_fastvit_stage(model, stage_idx):
    assert 0 <= stage_idx < 4, "Invalid FastViT stage index"
    try:
        # Access stages via index
        target_stage = model.backbone.backbone.stages[stage_idx]
        print(f"🔓 Unfreezing FastViT stage {stage_idx}")
        for p in target_stage.parameters():
            p.requires_grad = True
    except (AttributeError, KeyError) as e:
        print(f"Error accessing stage: {e}")

def stage_wise_unfreeze(model, epoch):
    """
    Progressive unfreezing schedule
    """

    # Epoch → action map
    if epoch == 0:
        freeze_all_backbone(model)

    elif epoch == 10:
        unfreeze_fastvit_stage(model, stage_idx=3)  # highest level

    elif epoch == 15:
        unfreeze_fastvit_stage(model, stage_idx=2)

    elif epoch == 20:
        unfreeze_fastvit_stage(model, stage_idx=1)

    elif epoch == 25:
        unfreeze_fastvit_stage(model, stage_idx=0)

# =========================================================
# Smoke Test
# =========================================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on {device}")

    # Build model and move to device
    model = build_model(num_classes=6).to(device)
    model.eval()

    # Create dummy images on the same device
    images = [
        torch.randn(3, 640, 640).to(device),
        torch.randn(3, 640, 640).to(device)
    ]

    try:
        with torch.no_grad():
            outputs = model(images)
        print("✅ OK – Output generated successfully")
        print(f"   Detections in first image: {len(outputs[0]['boxes'])}")
    except Exception as e:
        print(f"❌ Error during forward pass: {e}")

Running on cuda
2026-02-25 09:52:09,697 | INFO | Loading pretrained weights from Hugging Face hub (timm/fastvit_t8.apple_in1k)
2026-02-25 09:52:09,874 | INFO | HTTP Request: HEAD https://huggingface.co/timm/fastvit_t8.apple_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-02-25 09:52:09,992 | INFO | HTTP Request: GET https://huggingface.co/api/models/timm/fastvit_t8.apple_in1k/xet-read-token/cbc6c66a249060c66b8236284c7c3998e900e944 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

2026-02-25 09:52:11,107 | INFO | [timm/fastvit_t8.apple_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-02-25 09:52:11,135 | INFO | Missing keys (head.fc.weight, head.fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
FastViT channels: [48, 96, 192, 384]
✅ OK – Output generated successfully
   Detections in first image: 100


# Model Config

In [24]:
import torch
import logging
from tabulate import tabulate
from torchmetrics.detection import MeanAveragePrecision
import csv
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
import os
# model setup
logging.info("\n" + "=" * 80)
logging.info("MODEL CONFIGURATION")
logging.info("=" * 80)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(num_classes =NUM_CLASSES).to(device)
model.to(device)

# Count parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

model_info = [
    ["Device", str(device)],
    ["Total Parameters", f"{total_params:,}"],
    ["Trainable Parameters", f"{trainable_params:,}"],
    ["Trainable Percentage", f"{trainable_params/total_params*100:.1f}%"],
    ["RPN", "Trainable"],
    ["ROI Heads", "Trainable"]
]

model_table = tabulate(model_info, tablefmt="grid")
for line in model_table.split('\n'):
    logging.info(line)

logging.info("=" * 80)

# =========================================================
# Layer-wise Parameter Summary (Compact)
# =========================================================
logging.info("\n" + "=" * 80)
logging.info("LAYER-WISE PARAMETER SUMMARY")
logging.info("=" * 80)

layer_stats = []

for name, param in model.named_parameters():
    layer_stats.append([
        name,
        list(param.shape),
        f"{param.numel():,}",
        "Yes" if param.requires_grad else "No"
    ])

layer_table = tabulate(
    layer_stats,
    headers=["Layer Name", "Shape", "Params", "Trainable"],
    tablefmt="grid"
)

# Log line by line (safe for large tables)
for line in layer_table.split("\n"):
    logging.info(line)

logging.info("=" * 80)



2026-02-25 09:52:12,946 | INFO | 
2026-02-25 09:52:12,947 | INFO | MODEL CONFIGURATION
2026-02-25 09:52:12,947 | INFO | ================================================================================
2026-02-25 09:52:13,029 | INFO | Loading pretrained weights from Hugging Face hub (timm/fastvit_t8.apple_in1k)
2026-02-25 09:52:13,115 | INFO | HTTP Request: HEAD https://huggingface.co/timm/fastvit_t8.apple_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-02-25 09:52:13,117 | INFO | [timm/fastvit_t8.apple_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-02-25 09:52:13,138 | INFO | Missing keys (head.fc.weight, head.fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
FastViT channels: [48, 96, 192, 384]
2026-02-25 09:52:13,424 | INFO | +----------------------+------------+
2026-02-25 09:52:13,424 | INFO | | Device               | cuda       |
2026-02-25 09:5

# Class weights

In [25]:
def compute_class_weights(dataset, class_map):
    counts = Counter()
    for _, targets in dataset:
        for label in targets['labels']:
            counts[label.item()] += 1

    total = sum(counts.values())
    weights = {}

    for cid, cnt in counts.items():
        weights[cid] = total / (len(counts) * cnt)

    return weights

class_weights = compute_class_weights(train_dataset, CLASS_MAP)

# Model, Optimizer, Metrics

In [26]:
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    params,
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

map_metric_05 = MeanAveragePrecision(
    iou_thresholds=[0.5],
    class_metrics=True
)     

# model metaa data

In [27]:
import torchvision
import sys

def build_checkpoint_metadata(config):
    return {
        "class_map": CLASS_MAP,                      # name -> id
        "id_to_class_name": ID_TO_CLASS_NAME,        # id -> name
        "num_classes": len(CLASS_MAP),
        "background_class_id": BACKGROUND_CLASS_ID,

        "metrics": {
            "map_iou_thresholds": [0.1, 0.3, 0.5],
            "primary_selection_metric": "mAP@0.5",
            "early_stopping_metric": "mAP@0.3",
        },

        "training_config": {
            "num_epochs": config.num_epochs,
            "batch_size": config.batch_size,
            "optimizer": optimizer.__class__.__name__,
            "initial_lr": optimizer.param_groups[0]["lr"],
            #"early_stop_patience": early_stop_patience,
        },

        "environment": {
            "python_version": sys.version,
            "torch_version": torch.__version__,
            "torchvision_version": torchvision.__version__,
            "device": str(device),
        }
    }


# Tranning Started

In [28]:
training_start_time = time.time()
print("=+="*20,"training time started","=+="*20)
print("="*10)
print(MAX_TRAINING_TIME)
print("="*10)
# =========================================================
# Imports
# =========================================================
import os
import csv
import logging
import numpy as np
import torch
from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# =========================================================
# Metrics (ONLY mAP@0.5)
# =========================================================
ID_TO_CLASS_NAME = {v: k for k, v in CLASS_MAP.items()}

map_metric = MeanAveragePrecision(
    iou_thresholds=[0.5],
    box_format="xyxy",
    class_metrics=False
)

map_metric_classwise = MeanAveragePrecision(
    iou_thresholds=[0.5],
    box_format="xyxy",
    class_metrics=True
)

best_map_05 = checkpoint.get("map_05", 0.0) if os.path.exists(config.final_path) else 0.0
best_epoch_05 = 0
min_delta= 0.00001

# =========================================================
# Output dirs
# =========================================================
metrics_dir = os.path.join(config.output_root, "metrics")
models_dir = os.path.join(config.output_root, "models")

os.makedirs(metrics_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

final_model_path = os.path.join(models_dir, "final_model.pth")
best_model_path = os.path.join(models_dir, "best_model.pth")

# =========================================================
# CSV init (class-wise AP@0.5)
# =========================================================
class_map_csv = os.path.join(metrics_dir, "classwise_map_05.csv")
with open(class_map_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "class_name", "class_id", "ap_05"])

# =========================================================
# CSV init (loss curves)
# =========================================================
loss_csv = os.path.join(metrics_dir, "loss_curves.csv")
with open(loss_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss"])


# =========================================================
# Helper functions
# =========================================================
def filter_predictions_only(preds, gts):
    filtered_preds = []
    for pred in preds:
        mask = pred["labels"] != BACKGROUND_CLASS_ID
        filtered_preds.append({
            "boxes": pred["boxes"][mask],
            "scores": pred["scores"][mask],
            "labels": pred["labels"][mask],
        })
    return filtered_preds, gts


def safe_extract_map_value(map_result, key):
    if key not in map_result or map_result[key] is None:
        return 0.0
    return max(map_result[key].item(), 0.0)
# =========================================================
# Resume or Start Fresh
# =========================================================
start_epoch = 0

if os.path.exists(config.final_path):
    print(f"🔁 Found existing model at {config.final_path}")
    checkpoint = torch.load(config.final_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint.get("epoch", 0)

    print(f"✅ Model loaded. Resuming from epoch {start_epoch}")

else:
    print("🚀 No existing model found. Starting training from scratch.")

# =========================================================
# Training loop
# =========================================================
logging.info("🚀 Start Training")

for epoch in range(start_epoch, config.num_epochs):

    #stage_wise_unfreeze(model, epoch)

    if epoch == 10:
        for g in optimizer.param_groups:
            g["lr"] = 1e-5

    # ------------------ TRAIN ------------------
    model.train()
    epoch_loss = 0.0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    train_loss_avg = epoch_loss / len(train_loader)
    logging.info(f"[Epoch {epoch+1}] Train Loss: {train_loss_avg:.4f}")

    # ------------------ VALIDATION ------------------
    model.eval()
    map_metric.reset()
    map_metric_classwise.reset()

    val_loss = 0.0

    with torch.no_grad():
        for images, targets in val_loader:

            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Compute validation loss
            model.train()
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            val_loss += loss.item()

            # Compute predictions
            model.eval()
            outputs = model(images)

            preds, gts = [], []
            for out, tgt in zip(outputs, targets):
                preds.append({
                    "boxes": out["boxes"].cpu(),
                    "scores": out["scores"].cpu(),
                    "labels": out["labels"].cpu()
                })
                gts.append({
                    "boxes": tgt["boxes"].cpu(),
                    "labels": tgt["labels"].cpu()
                })

            preds, gts = filter_predictions_only(preds, gts)

            valid = [(p, g) for p, g in zip(preds, gts) if len(g["boxes"]) > 0]
            if valid:
                p, g = zip(*valid)
                map_metric.update(list(p), list(g))
                map_metric_classwise.update(list(p), list(g))

    val_loss /= len(val_loader)

    # ------------------ METRICS ------------------
    map_res = map_metric.compute()
    map_res_class = map_metric_classwise.compute()

    val_map_05 = safe_extract_map_value(map_res, "map_50")

    logging.info(f"[Epoch {epoch+1}] Val Loss: {val_loss:.4f}")
    logging.info(f"[Epoch {epoch+1}] Val mAP@0.5: {val_map_05:.4f}")

    # ------------------ SAVE CSV ------------------
    with open(class_map_csv, "a", newline="") as f:
        writer = csv.writer(f)
        for cid, ap in zip(
            map_res_class["classes"].cpu().numpy(),
            map_res_class["map_per_class"].cpu().numpy()
        ):
            writer.writerow([
                epoch + 1,
                ID_TO_CLASS_NAME.get(int(cid), f"class_{cid}"),
                int(cid),
                round(float(ap), 6)
            ])

    with open(loss_csv, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch + 1,
            round(train_loss_avg, 6),
            round(val_loss, 6)
        ])
    # =========================================================
    # Time Limit Check (After Validation)
    # =========================================================
    elapsed_time = time.time() - training_start_time

    if elapsed_time >= MAX_TRAINING_TIME:
        print("⏰ 9-hour limit reached. Stopping training safely...")
        break
    if val_map_05 > best_map_05 + min_delta:
        best_map_05 = val_map_05
        best_epoch_05 = epoch + 1

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "map_05": best_map_05,
                "metadata": build_checkpoint_metadata(config)
            },
            best_model_path
        )

        logging.info(f"💾 Saved BEST model @ epoch {epoch+1} (mAP@0.5={best_map_05:.4f})")


# =========================================================
# Save FINAL model
# =========================================================
torch.save(
    {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "map_05": val_map_05,
        "metadata": build_checkpoint_metadata(config)
    },
    final_model_path
)

logging.info(f"📦 Final model saved to {final_model_path}")
print(f"Training finished | Best mAP@0.5 = {best_map_05:.4f} (epoch {best_epoch_05})")

=+==+==+==+==+==+==+==+==+==+==+==+==+==+==+==+==+==+==+==+= training time started =+==+==+==+==+==+==+==+==+==+==+==+==+==+==+==+==+==+==+==+=
32400
🚀 No existing model found. Starting training from scratch.
2026-02-25 09:52:35,164 | INFO | 🚀 Start Training
2026-02-25 10:03:33,902 | INFO | [Epoch 1] Train Loss: 0.3699
2026-02-25 10:03:49,672 | INFO | [Epoch 1] Val Loss: 0.2950
2026-02-25 10:03:49,674 | INFO | [Epoch 1] Val mAP@0.5: 0.6669
2026-02-25 10:03:50,233 | INFO | 💾 Saved BEST model @ epoch 1 (mAP@0.5=0.6669)
2026-02-25 10:14:46,473 | INFO | [Epoch 2] Train Loss: 0.2254
2026-02-25 10:15:02,196 | INFO | [Epoch 2] Val Loss: 0.2685
2026-02-25 10:15:02,197 | INFO | [Epoch 2] Val mAP@0.5: 0.7696
2026-02-25 10:15:03,002 | INFO | 💾 Saved BEST model @ epoch 2 (mAP@0.5=0.7696)
2026-02-25 10:25:58,441 | INFO | [Epoch 3] Train Loss: 0.1920
2026-02-25 10:26:14,092 | INFO | [Epoch 3] Val Loss: 0.2718
2026-02-25 10:26:14,094 | INFO | [Epoch 3] Val mAP@0.5: 0.7726
2026-02-25 10:26:14,898 | IN

# necessary functions for testing 

In [29]:
# =========================================================
# Imports
# =========================================================
import os
import csv
import logging
import numpy as np
import matplotlib.pyplot as plt
import torch

from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision


# =========================================================
# Class helpers
# =========================================================
ID_TO_CLASS_NAME = {v: k for k, v in CLASS_MAP.items()}
NUM_CLASSES = len(ID_TO_CLASS_NAME)


# =========================================================
# SINGLE TEST DIRECTORY
# =========================================================
test_dir = os.path.join("/kaggle/working/", "test")
os.makedirs(test_dir, exist_ok=True)


# =========================================================
# Metric
# =========================================================
test_map_metric = MeanAveragePrecision(
    iou_thresholds=[0.5],
    box_format="xyxy",
    class_metrics=True
)


# =========================================================
# Helpers
# =========================================================
def filter_predictions_only(preds, gts):
    filtered_preds = []
    for p in preds:
        mask = p["labels"] != BACKGROUND_CLASS_ID
        filtered_preds.append({
            "boxes": p["boxes"][mask],
            "scores": p["scores"][mask],
            "labels": p["labels"][mask],
        })
    return filtered_preds, gts


def safe_extract_map_value(res, key):
    if key not in res or res[key] is None:
        return 0.0
    try:
        return max(res[key].item(), 0.0)
    except Exception:
        return 0.0

def compute_ap_from_pr(recall, precision):
    """
    101-point interpolated AP (COCO-style integration)
    """
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])
    return ap

# =========================================================
# Precision–Recall Curve (per class)
# =========================================================
def compute_pr_curve_single_class(preds, gts, class_id, iou_thresh=0.5):
    scores = []
    matches = []
    total_gt = 0

    for pred, gt in zip(preds, gts):

        p_mask = pred["labels"] == class_id
        g_mask = gt["labels"] == class_id

        p_boxes = pred["boxes"][p_mask]
        p_scores = pred["scores"][p_mask]
        g_boxes = gt["boxes"][g_mask]

        total_gt += len(g_boxes)

        if len(p_boxes) == 0:
            continue

        order = torch.argsort(p_scores, descending=True)
        p_boxes = p_boxes[order]
        p_scores = p_scores[order]

        matched_gt = torch.zeros(len(g_boxes), dtype=torch.bool)

        ious = box_iou(p_boxes, g_boxes) if len(g_boxes) > 0 else None

        for i in range(len(p_boxes)):
            scores.append(p_scores[i].item())

            if ious is None or len(g_boxes) == 0:
                matches.append(0)
                continue

            ious_i = ious[i].clone()
            ious_i[matched_gt] = -1.0

            max_iou, idx = ious_i.max(0)

            if max_iou >= iou_thresh:
                matches.append(1)
                matched_gt[idx] = True
            else:
                matches.append(0)

    if total_gt == 0 or len(matches) == 0:
        return None, None

    order = np.argsort(scores)[::-1]
    matches = np.array(matches)[order]

    tps = np.cumsum(matches)
    fps = np.cumsum(1 - matches)

    recall = tps / (total_gt + 1e-6)
    precision = tps / (tps + fps + 1e-6)

    return recall, precision
    
def plot_combined_pr_curve(preds, gts, save_path):
    plt.figure(figsize=(9, 7))

    ap_list = []

    for cid, name in ID_TO_CLASS_NAME.items():
        if cid == BACKGROUND_CLASS_ID:
            continue

        recall, precision = compute_pr_curve_single_class(
            preds, gts, cid, iou_thresh=0.5
        )

        if recall is None:
            continue

        ap = compute_ap_from_pr(recall, precision)
        ap_list.append(ap)

        plt.plot(
            recall,
            precision,
            linewidth=1.8,
            label=f"{name} (AP@0.5 = {ap:.4f})"
        )

    # ---- Compute mAP ----
    if len(ap_list) > 0:
        mAP = np.mean(ap_list)
    else:
        mAP = 0.0

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision–Recall Curve (mAP@0.5 = {mAP:.4f})")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend(loc="lower left", fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"\n Per-class AP@0.5:")
    for cid, name in ID_TO_CLASS_NAME.items():
        if cid == BACKGROUND_CLASS_ID:
            continue
        # recompute for printing clarity
        recall, precision = compute_pr_curve_single_class(
            preds, gts, cid, iou_thresh=0.5
        )
        if recall is not None:
            ap = compute_ap_from_pr(recall, precision)
            print(f"{name}: {ap:.4f}")

    print(f"\n mAP@0.5: {mAP:.4f}")


def compute_detection_confusion_matrix(preds, gts, iou_thresh=0.5):
    """
    Confusion matrix including:
    - TP (correct matches)
    - FP (prediction but no GT)
    - FN (GT but no prediction)

    Row = GT class
    Column = Predicted class
    Background class = 0
    """
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    for pred, gt in zip(preds, gts):

        p_boxes = pred["boxes"]
        p_labels = pred["labels"]
        g_boxes = gt["boxes"]
        g_labels = gt["labels"]

        matched_gt = np.zeros(len(g_boxes), dtype=bool)

        if len(p_boxes) > 0 and len(g_boxes) > 0:
            ious = box_iou(p_boxes, g_boxes)

            order = torch.argsort(pred["scores"], descending=True)

            for i in order:
                i = i.item()
                max_iou, idx = ious[i].max(0)

                if max_iou >= iou_thresh and not matched_gt[idx]:
                    gt_cls = int(g_labels[idx])
                    pr_cls = int(p_labels[i])
                    cm[gt_cls, pr_cls] += 1
                    matched_gt[idx] = True
                else:
                    # False Positive
                    pr_cls = int(p_labels[i])
                    cm[0, pr_cls] += 1

        else:
            # All predictions are FP
            for pr_cls in p_labels:
                cm[0, int(pr_cls)] += 1

        # False Negatives (unmatched GT)
        for idx, matched in enumerate(matched_gt):
            if not matched:
                gt_cls = int(g_labels[idx])
                cm[gt_cls, 0] += 1

    return cm

def plot_confusion_matrix(cm, save_path, normalize=True):
    if normalize:
        cm = cm.astype(np.float32)
        cm = cm / (cm.sum(axis=1, keepdims=True) + 1e-6)

    plt.figure(figsize=(8, 7))
    plt.imshow(cm, cmap="Blues")  # 🔵 WHITE → BLUE
    plt.title("Normalized Detection Confusion Matrix (IoU ≥ 0.5)")
    plt.colorbar()

    classes = [ID_TO_CLASS_NAME[i] for i in range(NUM_CLASSES)]
    ticks = np.arange(NUM_CLASSES)
    plt.xticks(ticks, classes, rotation=45, ha="right")
    plt.yticks(ticks, classes)

    thresh = cm.max() / 2.0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(
                j, i, f"{cm[i, j]:.2f}",
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    plt.xlabel("Predicted")
    plt.ylabel("Ground Truth")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

# Testing

In [30]:
# =========================================================
# CHECKPOINT LOADER
# =========================================================
def load_model_from_checkpoint(model, optimizer, checkpoint_path, device):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False  # 🔥 ADD THIS
    )

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    epoch = checkpoint.get("epoch", 0)
    map_05 = checkpoint.get("map_05", 0.0)

    logging.info(f"📦 Loaded checkpoint from {checkpoint_path}")
    logging.info(f"   → Epoch: {epoch}")
    logging.info(f"   → mAP@0.5: {map_05:.4f}")

    return model


# =========================================================
# FULL TEST EVALUATION FUNCTION
# =========================================================
def evaluate_model_on_test(
    model,
    test_loader,
    device,
    save_root_dir,
    model_tag="model"
):
    os.makedirs(save_root_dir, exist_ok=True)

    metric = MeanAveragePrecision(
        iou_thresholds=[0.5],
        box_format="xyxy",
        class_metrics=True
    )

    model.eval()
    metric.reset()

    all_preds, all_gts = [], []

    logging.info(f"🧪 Running Test Evaluation → {model_tag}")

    with torch.no_grad():
        for images, targets in test_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            preds, gts = [], []

            for out, tgt in zip(outputs, targets):
                preds.append({
                    "boxes": out["boxes"].cpu(),
                    "scores": out["scores"].cpu(),
                    "labels": out["labels"].cpu()
                })
                gts.append({
                    "boxes": tgt["boxes"].cpu(),
                    "labels": tgt["labels"].cpu()
                })

            metric.update(preds, gts)
            all_preds.extend(preds)
            all_gts.extend(gts)

    res = metric.compute()
    map_50 = res["map_50"].item()

    logging.info(f"✅ {model_tag} | mAP@0.5 = {map_50:.4f}")

    # Save per-class AP
    csv_path = os.path.join(save_root_dir, f"{model_tag}_classwise_map_05.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class_name", "class_id", "map_05"])
        for cid, ap in zip(
            res["classes"].cpu().numpy(),
            res["map_per_class"].cpu().numpy()
        ):
            writer.writerow([
                ID_TO_CLASS_NAME[int(cid)],
                int(cid),
                round(float(ap), 6)
            ])

    # PR Curve
    plot_combined_pr_curve(
        all_preds,
        all_gts,
        save_path=os.path.join(save_root_dir, f"{model_tag}_precision_recall_curve.png")
    )

    # Confusion Matrix
    cm = compute_detection_confusion_matrix(all_preds, all_gts)

    plot_confusion_matrix(
        cm,
        save_path=os.path.join(save_root_dir, f"{model_tag}_confusion_matrix.png"),
        normalize=True
    )

    return map_50

test_results_root = "/kaggle/working/test_results"
os.makedirs(test_results_root, exist_ok=True)



# =========================================================
# 2️⃣ Evaluate FINAL model
# =========================================================
model = load_model_from_checkpoint(
    model,
    optimizer=None,
    checkpoint_path=final_model_path,
    device=device
)

final_map = evaluate_model_on_test(
    model=model,
    test_loader=test_loader,
    device=device,
    save_root_dir=os.path.join(test_results_root, "final_model"),
    model_tag="final_model"
)

# =========================================================
# 1️⃣ Evaluate BEST model
# =========================================================
model = load_model_from_checkpoint(
    model,
    optimizer=None,
    checkpoint_path=best_model_path,
    device=device
)

best_map = evaluate_model_on_test(
    model=model,
    test_loader=test_loader,
    device=device,
    save_root_dir=os.path.join(test_results_root, "best_model"),
    model_tag="best_model"
)

logging.info(f"🏁 BEST mAP@0.5  = {best_map:.4f}")
logging.info(f"🏁 FINAL mAP@0.5 = {final_map:.4f}")


2026-02-25 19:00:33,040 | INFO | 📦 Loaded checkpoint from models/final_model.pth
2026-02-25 19:00:33,041 | INFO |    → Epoch: 49
2026-02-25 19:00:33,042 | INFO |    → mAP@0.5: 0.8080
2026-02-25 19:00:33,048 | INFO | 🧪 Running Test Evaluation → final_model
2026-02-25 19:00:41,795 | INFO | ✅ final_model | mAP@0.5 = 0.7844

 Per-class AP@0.5:
black_border: 0.9003
broken: 0.6177
hot_spot: 0.8955
no_electricity: 0.9761
scratch: 0.5396

 mAP@0.5: 0.7858
2026-02-25 19:00:43,604 | INFO | 📦 Loaded checkpoint from models/best_model.pth
2026-02-25 19:00:43,605 | INFO |    → Epoch: 35
2026-02-25 19:00:43,606 | INFO |    → mAP@0.5: 0.8140
2026-02-25 19:00:43,613 | INFO | 🧪 Running Test Evaluation → best_model
2026-02-25 19:00:52,336 | INFO | ✅ best_model | mAP@0.5 = 0.7718

 Per-class AP@0.5:
black_border: 0.8969
broken: 0.5908
hot_spot: 0.8975
no_electricity: 0.9167
scratch: 0.5747

 mAP@0.5: 0.7753
2026-02-25 19:00:53,723 | INFO | 🏁 BEST mAP@0.5  = 0.7718
2026-02-25 19:00:53,724 | INFO | 🏁 FINAL 

# zip

In [31]:
import os
import shutil

root_path = '/kaggle/working/'
os.chdir(root_path)

files_in_root = []

for item in os.listdir('.'):
    if item.startswith('.'):
        continue
    
    if os.path.isdir(item):
        # 1. Zip the directory
        shutil.make_archive(item, 'zip', item)
        # 2. Delete the original directory
        shutil.rmtree(item)
        print(f"Zipped and deleted folder: {item}")
        
    elif os.path.isfile(item):
        # --- UPDATED CODE START ---
        # Skip existing zips AND skip .pth files
        if not item.endswith('.zip') and not item.endswith('.pth'):
            files_in_root.append(item)
        # --- UPDATED CODE END ---

# 3. Handle root files: Zip them into root.zip if any exist
if files_in_root:
    temp_dir = 'root_files_temp'
    os.makedirs(temp_dir, exist_ok=True)
    
    for f in files_in_root:
        shutil.move(f, os.path.join(temp_dir, f))
    
    shutil.make_archive('root', 'zip', temp_dir)
    shutil.rmtree(temp_dir)
    print("Zipped and deleted root level files into root.zip (excluding .pth files)")

Zipped and deleted folder: test_results
Zipped and deleted folder: metrics
Zipped and deleted folder: test
Zipped and deleted folder: final_dataset
Zipped and deleted folder: logs
Zipped and deleted folder: models
Zipped and deleted root level files into root.zip (excluding .pth files)
